# Data Request Service

This notebook explains how to use the **Data Request Service** — the tool that sits between the lead parameters (@Rodrigo) and the ETL (@Gabriel).

```
Lead params (peril, AOI, window)
         ↓
  DataRequestService        ← you are here
         ↓
  .nc files  +  summary.json
         ↓
    S3 bucket  →  ETL (@Gabriel)
```

---

### What it does
1. Takes a **peril** + **AOI** as input
2. Auto-selects the right data sources and variables
3. Downloads climate data (ERA5, ERA5-Land, CHIRPS) for the standard historical period
4. Saves one `.nc` file per year, locally and optionally in S3
5. Writes a `summary.json` with everything that was downloaded

## 0. Setup

In [ ]:
from piscis import (
    DataRequestService,
    DataRequest,
    list_perils,
    compute_period,
)

---
## 1. What perils are available?

Each peril maps to a fixed set of data sources and climate variables.
You don't need to know which variables to download — the service handles that.

In [ ]:
from piscis import get_peril_config

for name in list_perils():
    cfg = get_peril_config(name)
    print(f"  {name}")
    print(f"    → {cfg.description}")
    for s in cfg.sources:
        print(f"    → source: {s.source_type}  |  vars: {s.variables}")
    print()

---
## 2. What period will be downloaded?

By default the service picks the **most recent 30-year window** that starts at a multiple of 5.

| Reference year | Period |
|---|---|
| 2026 | 1995 – 2025 |
| 2024 | 1990 – 2020 |
| 2021 | 1990 – 2020 |

You can always override this manually.

In [ ]:
# Auto-computed (default)
start, end = compute_period()
print(f"Default period: {start}–{end}")

# 20-year fallback (when data availability is limited)
start20, end20 = compute_period(window=20)
print(f"20-year window: {start20}–{end20}")

---
## 3. How to specify an AOI

Two options: **bounding box** (dict) or a **shapefile path**.

In [ ]:
from piscis import parse_aoi

# Option A — bounding box dict  (maxy=North, miny=South, minx=West, maxx=East)
aoi_bbox = {
    "maxy":  6.3,
    "miny":  1.6,
    "minx": -75.0,
    "maxx": -69.9,
}

# Option B — shapefile path  (service reprojects to WGS84 automatically)
# aoi_shp = "data/shapefiles/region.shp"

print(parse_aoi(aoi_bbox))

---
## 4. Run a request — the simple way

This is the most common call. The service figures out period, sources, and variables automatically.

In [ ]:
result = DataRequestService().run(
    DataRequest(
        lead_id = "lead_001",
        peril   = "drought",
        aoi     = {"maxy": 6.3, "miny": 1.6, "minx": -75.0, "maxx": -69.9},
        # period  = (1995, 2025),   ← uncomment to override
        # s3_bucket = "suyana-climate-data",  ← uncomment to upload to S3
    )
)

---
## 5. Reading the result

The result object has everything you need.

In [ ]:
print("Status  :", "✓ success" if result.success else "⚠ partial" if result.partial else "✗ failed")
print("Period  :", result.period)
print("Files   :", len(result.nc_files))
for f in result.nc_files:
    print("  ", f)
print("Summary :", result.summary_path)

In [ ]:
# The summary dict has the full metadata — useful for Gabriel's ETL
import json
print(json.dumps(result.summary, indent=2))

---
## 6. Run from a YAML file

For more structured workflows (e.g. one YAML per lead), use `run_from_yaml`.

The template is at `configs/request_template.yml` — copy it, fill in the fields, and run.

In [ ]:
# result = DataRequestService().run_from_yaml("configs/request_template.yml")

The YAML looks like this:

```yaml
lead_id: "lead_001"
peril:   "drought"

aoi:
  maxy:  6.3
  miny:  1.6
  minx: -75.0
  maxx: -69.9

# period:              # auto-computed if omitted
#   start_year: 1995
#   end_year:   2025

output_dir: "data/requests"
s3_bucket:  "suyana-climate-data"   # leave blank to skip
```

---
## 7. With S3 upload

Add `s3_bucket` to any request. The bucket is created automatically if it doesn't exist, with AES-256 encryption and public access blocked.

In [ ]:
# result = DataRequestService().run(
#     DataRequest(
#         lead_id   = "lead_001",
#         peril     = "drought",
#         aoi       = {"maxy": 6.3, "miny": 1.6, "minx": -75.0, "maxx": -69.9},
#         s3_bucket = "suyana-climate-data",
#         s3_region = "us-east-1",
#         # s3_prefix = "leads/lead_001/drought"  ← auto-set if blank
#     )
# )

# print(result.s3_uris)   # → ["s3://suyana-climate-data/leads/lead_001/drought/era5land.1995.nc", ...]

---
## 8. How to add a new peril or data source

The service is designed to be extended one piece at a time.

### Add a new peril  →  `piscis/peril_config.py`

Open `piscis/peril_config.py` and add an entry to `PERIL_CONFIGS`:

```python
"wind" : PerilConfig(
    peril       = "wind",
    description = "Strong wind events",
    sources = [
        SourceConfig(
            source_type = "era5",
            dataset     = "reanalysis-era5-single-levels",
            variables   = ["10m_u_component_of_wind", "10m_v_component_of_wind"],
            hours       = ["00", "06", "12", "18"],
        )
    ],
),
```

That's it — no other changes needed if the source type is already supported (`era5`, `era5_land`, `chirps`).

---

### Add a new data source  →  3 steps

**Step 1** — Create `piscis/chirts_downloader.py` (or whichever source).
It needs to look like this:

```python
class CHIRTSDownloader:
    def __init__(self, aoi: BoundingBox, output_dir: str): ...
    def download_period(self, start_year: int, end_year: int) -> dict:
        # must return {"files": [...], "failed": [...]}
        ...
```

**Step 2** — Uncomment the block in `piscis/service.py → _dispatch_download()`:

```python
elif source.source_type == "chirts":
    from .chirts_downloader import CHIRTSDownloader
    dl = CHIRTSDownloader(aoi=aoi, output_dir=output_dir)
    return dl.download_period(start_year=start_year, end_year=end_year)
```

**Step 3** — Add `source_type: "chirts"` to any peril config in `piscis/peril_config.py`.

---
## 9. File structure

```
climate_data_retrieval/
│
├── piscis/
│   ├── service.py           ← DataRequestService (orchestrator)
│   ├── peril_config.py      ← peril → source + variables mapping
│   ├── aoi.py               ← BoundingBox, parse shapefile
│   ├── period.py            ← 30-year period calculator
│   ├── era5_downloader.py   ← ERA5 / ERA5-Land (CDS API)
│   ├── chirps_downloader.py ← CHIRPS v3.0 (CHC/UCSB)
│   ├── s3_storage.py        ← upload to S3 (AES-256, no public access)
│   └── summary.py           ← JSON summary generator
│
└── configs/
    ├── request_template.yml ← copy this for each new lead
    └── perils/
        ├── drought.yml
        ├── heatwave.yml
        ├── cold_spell.yml
        └── precipitation.yml
```